# S0-03 `@hobo/core` draft POC — HOBO-580

The core package is a **parse-don't-validate** boundary: raw JSON (event envelopes, port
descriptors, capability-manifest bundles) either parses into a **deep-frozen typed record**
or throws a typed `CoreParseError` — no half-parsed values escape. Below, each demonstration
runs against the package's real committed fixtures under `test/fixtures/`.

In [1]:
import { strict as assert } from "node:assert";
import { readFileSync } from "node:fs";
import { parseEventEnvelope, toCloudEvent, fromCloudEvent, EnvelopeParseError } from "./src/index.ts";

const fixture = (name: string): unknown =>
  JSON.parse(readFileSync(new URL(`./test/fixtures/${name}`, import.meta.url), "utf8"));

// The committed valid fixture parses into a frozen record; CloudEvent round-trip is lossless.
const envelope = parseEventEnvelope(fixture("valid-envelope.json"));
console.log(`envelope: type=${envelope.type} seq=${envelope.seq} frozen=${Object.isFrozen(envelope)}`);
const cloudEvent = toCloudEvent(envelope);
console.log(`cloudevent source=${cloudEvent.source}`);
assert.deepEqual(fromCloudEvent(cloudEvent), envelope);
console.log("round-trip lossless: true");

// Malformed committed fixtures are rejected with typed, path-bearing errors.
for (const name of ["malformed-envelope.json", "malformed-envelope-actor.json"]) {
  try {
    parseEventEnvelope(fixture(name));
  } catch (error) {
    assert.ok(error instanceof EnvelopeParseError);
    console.log(`rejected ${name}: path=${error.path} expected=${error.expected}`);
  }
}

envelope: type=identity.Issued seq=7 frozen=true
cloudevent source=hobo://tenant-acme/identity/svid-agent-1
round-trip lossless: true
rejected malformed-envelope.json: path=streamId expected=non-empty string
rejected malformed-envelope-actor.json: path=actor.kind expected=user | agent | service | system


## Ports and the container

Port descriptors require WIT-style interface refs (`ns:pkg/iface@semver`); the DI container
keys on `kind:interface:name` and refuses duplicate registrations.

In [2]:
import { strict as assert } from "node:assert";
import { createContainer, createPortDescriptor, PortDescriptorParseError } from "./src/index.ts";

const cachePort = createPortDescriptor({
  kind: "port", name: "cache", interface: "hobo:data/cache@1.0.0", stability: "preview",
});
console.log(`descriptor: ${cachePort.kind}/${cachePort.name} -> ${cachePort.interface}`);

try {
  createPortDescriptor({ kind: "port", name: "bad", interface: "not a wit ref", stability: "stable" });
} catch (error) {
  assert.ok(error instanceof PortDescriptorParseError);
  console.log("non-WIT interface ref rejected: true");
}

const container = createContainer();
container.register(cachePort, { name: "sim-cache" });
assert.deepEqual(container.resolve(cachePort), { name: "sim-cache" });
assert.throws(() => container.register(cachePort, {}), /already registered/);
console.log("container: resolve ok, duplicate registration rejected");

descriptor: port/cache -> hobo:data/cache@1.0.0
non-WIT interface ref rejected: true
container: resolve ok, duplicate registration rejected


## Capability-manifest bundles

The 4-file provider manifest fixture (`index` + `capabilities` + `auth` + `cost`) parses as a
unit; cross-file invariants (auth issuer must match index issuer) fail the whole bundle.

In [3]:
import { strict as assert } from "node:assert";
import { readFileSync } from "node:fs";
import { parseCapabilityManifestBundle, ManifestParseError } from "./src/index.ts";

const fixture = (name: string): unknown =>
  JSON.parse(readFileSync(new URL(`./test/fixtures/${name}`, import.meta.url), "utf8"));

const bundle = parseCapabilityManifestBundle({
  index: fixture("valid-manifest-bundle/index.json"),
  capabilities: fixture("valid-manifest-bundle/capabilities.json"),
  auth: fixture("valid-manifest-bundle/auth.json"),
  cost: fixture("valid-manifest-bundle/cost.json"),
});
console.log(`issuer=${bundle.index.issuer}`);
console.log(`ports=${bundle.capabilities.ports.map((p) => p.port).join(",")} flows=${bundle.auth.flows.map((f) => f.type).join(",")}`);

try {
  parseCapabilityManifestBundle(fixture("malformed-manifest-issuer.json"));
} catch (error) {
  assert.ok(error instanceof ManifestParseError);
  console.log(`rejected: path=${error.path} expected=${error.expected}`);
}
console.log("CONTRACT HELD: core parses-not-validates into frozen typed records");

issuer=https://acme-cloud.example.com
ports=compute,database flows=oidc-federation,api-key
rejected: path=auth.issuer expected=match index.issuer
CONTRACT HELD: core parses-not-validates into frozen typed records
